In [0]:
df=spark.read.table("dbws01.default.sales_orders")

In [0]:
%sql 
create schema if not exists dbws01.delta;

In [0]:
%sql 
create volume if not exists dbws01.delta.sales_orders_delta;

In [0]:
#loading the data into a delta table
delta_table_path="/Volumes/dbws01/delta/sales_orders_delta"
df.write.format('delta').save(delta_table_path)

In [0]:
df.show(2)

+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesorderLineNumber| OrderDate|CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01| Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|  Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+
only showing top 2 rows


In [0]:
from delta.tables import * 
from pyspark.sql.functions import *

deltatable=DeltaTable.forPath(spark, delta_table_path)

deltatable.update(condition=col("SalesOrderNumber")==lit('SO43701'), set={"OrderDate": to_date(lit("2020-01-01"))})

DataFrame[num_affected_rows: bigint]

In [0]:
deltatable.toDF().filter(col('SalesOrderNumber')==lit('SO43701')).show()

+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesorderLineNumber| OrderDate|CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2020-01-01| Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
+----------------+--------------------+----------+------------+--------------------+--------------------+--------+---------+--------+



In [0]:
new_df=spark.read.format('delta').load(delta_table_path)

In [0]:
#auditablilty
deltatable.history(10).show()

+-------+-------------------+---------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|         userId|            userName|operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+---------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      2|2026-09-27 19:35:28|146169615611989|satyakeb@omnics12...|   UPDATE|{predicate -> ["(...|NULL|{1290639618232442}|   c30d1280-d644-418...|0927-191341-i3egr...|          1|WriteSerializable|      

In [0]:
#create a data catalog table
df.write.format('delta').saveAsTable("delta.sales_orders_delta")

In [0]:
%sql 
select * from dbws01.delta.sales_orders_delta

SalesOrderNumber,SalesorderLineNumber,OrderDate,CustomerName,Email,Item,Quantity,UnitPrice,Tax
SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992
SO43700,1,2019-07-01,Ruben Prasad,ruben10@adventure-works.com,"Road-650 Black, 62",1,699.0982,55.9279
SO43703,1,2019-07-01,Albert Alvarez,albert7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
SO43697,1,2019-07-01,Cole Watson,cole1@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
SO43699,1,2019-07-01,Sydney Wright,sydney61@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43702,1,2019-07-01,Colin Anand,colin45@adventure-works.com,"Road-150 Red, 44",1,3578.27,286.2616
SO43698,1,2019-07-01,Rachael Martinez,rachael16@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43707,1,2019-07-02,Emma Brown,emma3@adventure-works.com,"Road-150 Red, 48",1,3578.27,286.2616
